---

# SundaLife

*Visualisasi Data Kemiskinan Jawa Barat 2021-2025*

---

## Kelompok 01 - IF4061 Visualisasi Data

- 13523004 - Razi Rachman Widyadhana
- 13523006 - William Andrian Dharma T
- 13523086 - Bob Kunanda
- 13523103 - Steven Owen Liauw
- 13523109 - Haegen Quinston

---

## Daftar Isi

1. [**Pendahuluan**](#1)
2. [**Inisialisasi**](#2)
3. [**Persiapan Data**](#3)
4. [**Transformasi Data**](#4)
5. [**Grafik 1: Tren Kemiskinan Pulau Jawa 2021-2025**](#5)
6. [**Grafik 2: Peta Sebaran Kemiskinan Jawa Barat**](#6)
7. [**Grafik 3: Korelasi Kemiskinan dan Pengangguran**](#7)
8. [**Grafik 4: Kota-Kota Anomali**](#8)


---

# Pendahuluan <a name="1"></a>

---

Jawa Barat merupakan provinsi dengan jumlah penduduk terbesar di Indonesia, namun menyimpan disparitas kemiskinan yang signifikan antarwilayah. Pola kemiskinan di tingkat kabupaten/kota memperlihatkan dinamika yang berbeda dari gambaran agregat provinsi, dan perlu ditelusuri secara lebih rinci.

Analisis ini menelusuri empat pertanyaan: bagaimana posisi Jawa Barat dibandingkan provinsi lain di Pulau Jawa selama 2021-2025, di mana kemiskinan paling terkonsentrasi di tingkat kabupaten/kota, seberapa kuat hubungan antara kemiskinan dan pengangguran, serta wilayah mana yang memperlihatkan pola berbeda dari mayoritas.

---

# Inisialisasi <a name="2"></a>

---

`geopandas` digunakan untuk membaca batas wilayah dari file GeoJSON GADM 4.1 dan memplot peta koropleth. Seluruh impor diletakkan dalam satu sel.

In [ ]:
# %pip install geopandas --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats

%matplotlib inline

KAGGLE_DATA_DIR = '/kaggle/input/datasets/stevenowen/data-vizdat/data/'
if os.path.exists(KAGGLE_DATA_DIR):
    DATA_DIR = KAGGLE_DATA_DIR
    RAW_DIR = DATA_DIR + 'raw/'
    CLEAN_DIR = '/kaggle/working/data/clean/'
    OUTPUT_DIR = '/kaggle/working/output/'
    GADM_PATH = DATA_DIR + 'clean/gadm41_IDN_2.json'
else:
    DATA_DIR = '../data/'
    RAW_DIR = DATA_DIR + 'raw/'
    CLEAN_DIR = DATA_DIR + 'clean/'
    OUTPUT_DIR = '../docs/output/'
    GADM_PATH = CLEAN_DIR + 'gadm41_IDN_2.json'

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'figure.facecolor': 'none',
    'axes.facecolor':   'none',
})

---

# Persiapan Data <a name="3"></a>

---

Tahap ini membaca data mentah dari BPS dan menghasilkan dua file bersih di `data/clean/`. Seluruh proses dibuat bertahap agar setiap keluaran pada Bagian 2 dokumen dapat diambil sebagai screenshot tanpa mengubah alur analisis utama.

## Bagian 2.2.1 dan 2.4.1 - Dataset ppo Tahunan

Cell berikut menampilkan pratinjau file mentah `ppo_indonesia_2025.csv` untuk Gambar 2.2.1, lalu contoh hasil filter enam provinsi Pulau Jawa dan seleksi nilai semester untuk Gambar 2.4.1.

In [ ]:
YEARS = [2021, 2022, 2023, 2024, 2025]
JAWA  = [
    'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR',
    'DKI JAKARTA', 'BANTEN', 'DI YOGYAKARTA',
]

ppo_raw_2025 = pd.read_csv(RAW_DIR + 'ppo_indonesia_2025.csv', skiprows=4)
ppo_raw_2025.head()

In [ ]:
ppo_example = ppo_raw_2025.iloc[:, [0, 1, 2]].copy()
ppo_example.columns = ['Provinsi', 'S1', 'S2']
ppo_example['Provinsi'] = ppo_example['Provinsi'].str.strip()
ppo_example = ppo_example[ppo_example['Provinsi'].isin(JAWA)].copy()
ppo_example['S1'] = pd.to_numeric(ppo_example['S1'], errors='coerce')
ppo_example['S2'] = pd.to_numeric(ppo_example['S2'], errors='coerce')
ppo_example['2025'] = ppo_example['S2'].fillna(ppo_example['S1'])
ppo_example

## Bagian 2.5.1 - Penggabungan Lima Tahun ke Format Wide

Setelah fungsi pemrosesan per tahun didefinisikan, semua file tahunan digabungkan menjadi `ppo_jawa_2021-2025.csv`.

In [ ]:
def process_ppo_year(year):
    df = pd.read_csv(RAW_DIR + f'ppo_indonesia_{year}.csv', skiprows=4)
    df = df.iloc[:, [0, 1, 2]]
    df.columns = ['Provinsi', 'S1', 'S2']
    df['Provinsi'] = df['Provinsi'].str.strip()
    df = df[df['Provinsi'].isin(JAWA)].copy()
    df['S1'] = pd.to_numeric(df['S1'], errors='coerce')
    df['S2'] = pd.to_numeric(df['S2'], errors='coerce')
    df[str(year)] = df['S2'].fillna(df['S1'])
    return df[['Provinsi', str(year)]]

processed = [process_ppo_year(year) for year in YEARS]
ppo_jawa = processed[0]
for df_year in processed[1:]:
    ppo_jawa = ppo_jawa.merge(df_year, on='Provinsi', how='outer')

ppo_jawa.to_csv(CLEAN_DIR + 'ppo_jawa_2021-2025.csv', index=False, encoding='utf-8')
ppo_jawa

## Bagian 2.2.2 - Dataset Kabupaten/Kota Jawa Barat Mentah

Pratinjau salah satu file indikator BPS kabupaten/kota ditampilkan sebelum pembersihan.

In [ ]:
POVERTY_PATH = RAW_DIR + 'Persentase_Penduduk_Miskin_Menurut_Kabupaten_Kota_di_Jawa_Barat_2025.csv'
UNEMPLOYMENT_PATH = RAW_DIR + 'Tingkat_Pengangguran_Terbuka_Menurut_Kabupaten_Kota_2025.csv'
POVERTY_LEVEL_PATH = RAW_DIR + 'Garis_Kemiskinan_Menurut_Kabupaten_Kota_2025.csv'

poverty_raw = pd.read_csv(POVERTY_PATH, header=None)
poverty_raw.head(10)

## Bagian 2.4.2 dan 2.4.4 - Cleaning dan Konversi Tipe Data

Tiga file indikator dibersihkan dengan fungsi yang sama. Output pertama menunjukkan hasil pembersihan satu indikator, sedangkan output kedua menunjukkan tipe data setelah konversi numerik.

In [ ]:
def clean_bps(path, col):
    df = pd.read_csv(path, header=None).iloc[:, :2]
    df.columns = ['region', col]
    df['region'] = df['region'].astype(str).str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['region', col])
    return df[~df['region'].str.contains('Provinsi', na=False)]

poverty_clean = clean_bps(POVERTY_PATH, 'poverty_rate')
unemployment_clean = clean_bps(UNEMPLOYMENT_PATH, 'unemployment_rate')
poverty_level_clean = clean_bps(POVERTY_LEVEL_PATH, 'poverty_level')

poverty_clean

In [ ]:
pd.DataFrame({
    'dataset': ['poverty_clean', 'unemployment_clean', 'poverty_level_clean'],
    'rows': [len(poverty_clean), len(unemployment_clean), len(poverty_level_clean)],
    'value_dtype': [poverty_clean['poverty_rate'].dtype,
                    unemployment_clean['unemployment_rate'].dtype,
                    poverty_level_clean['poverty_level'].dtype],
})

## Bagian 2.5.2 - Penggabungan Tiga Indikator

Ketiga indikator digabungkan dengan `region` sebagai key dan disimpan sebagai `jabar_2025_combined.csv`.

In [ ]:
jabar_raw = (
    poverty_clean
    .merge(unemployment_clean, on='region')
    .merge(poverty_level_clean, on='region')
)

jabar_raw.to_csv(CLEAN_DIR + 'jabar_2025_combined.csv', index=False, encoding='utf-8')
jabar_raw

---

# Transformasi Data <a name="4"></a>

---

Bagian ini menambahkan kolom turunan yang dipakai oleh visualisasi: `type`, `gadm_name`, `ratio`, kelompok anomali, dan peringkat. Output ditampilkan bertahap agar sesuai dengan penomoran Bagian 2 pada dokumen.

## Bagian 2.2.3 dan 2.4.3 - Pemeriksaan GADM dan Key Penggabungan

Cell pertama menampilkan contoh atribut `NAME_1`, `NAME_2`, dan `TYPE_2` dari GADM. Cell kedua menampilkan penyesuaian key wilayah dari data BPS ke `gadm_name`.

In [ ]:
gdf = gpd.read_file(GADM_PATH)
gdf_jabar_check = gdf[gdf['NAME_1'] == 'JawaBarat'].copy()
gdf_jabar_check[['NAME_1', 'NAME_2', 'TYPE_2']].drop_duplicates().head(12)

In [ ]:
ppo_jawa = pd.read_csv(CLEAN_DIR + 'ppo_jawa_2021-2025.csv')
jabar    = pd.read_csv(CLEAN_DIR + 'jabar_2025_combined.csv')

jabar = jabar.dropna(subset=['region'])
jabar['type'] = np.where(
    jabar['region'].str.startswith('Kota ', na=False), 'Kota', 'Kabupaten'
)

GADM_FIX = {'Bandung Barat': 'BandungBarat'}
jabar['gadm_name'] = jabar['region'].map(GADM_FIX).fillna(jabar['region'])

jabar[['region', 'type', 'gadm_name']].head(12)

## Bagian 2.5.3 - Penambahan Kolom Turunan

`ratio` dihitung dari `poverty_rate / unemployment_rate` untuk mengidentifikasi wilayah dengan pola kemiskinan relatif tinggi terhadap pengangguran.

In [ ]:
jabar['ratio'] = jabar['poverty_rate'] / jabar['unemployment_rate']
jabar[['region', 'type', 'gadm_name', 'poverty_rate',
       'unemployment_rate', 'poverty_level', 'ratio']]

## Bagian 2.5.4 - Penentuan Wilayah Anomali

Tiga rasio tertinggi dipilih setelah mengecualikan Majalengka, dan tiga rasio terendah dipilih sebagai pembanding.

In [ ]:
ANOMALI_HIGH = (
    jabar.nlargest(4, 'ratio')
    .query("region != 'Majalengka'")
    ['region'].head(3).tolist()
)
ANOMALI_LOW  = jabar.nsmallest(3, 'ratio')['region'].tolist()
ANOMALI_ALL  = ANOMALI_HIGH + ANOMALI_LOW

anomali_table = jabar[jabar['region'].isin(ANOMALI_ALL)].copy()
anomali_table['kelompok'] = np.where(
    anomali_table['region'].isin(ANOMALI_HIGH), 'Tinggi', 'Rendah'
)
anomali_table[['region', 'unemployment_rate', 'poverty_rate',
               'poverty_level', 'ratio', 'kelompok']].sort_values(
    ['kelompok', 'ratio'], ascending=[False, False]
)

## Bagian 2.5.5 - Penghitungan Peringkat

Peringkat TPT dan garis kemiskinan dihitung untuk seluruh wilayah, lalu ditampilkan untuk enam wilayah anomali.

In [ ]:
jabar['tpt_rank'] = jabar['unemployment_rate'].rank(method='min').astype(int)
jabar['gk_rank']  = jabar['poverty_level'].rank(method='min').astype(int)

anomali = jabar[jabar['region'].isin(ANOMALI_ALL)].copy()
anomali[['region', 'tpt_rank', 'gk_rank', 'unemployment_rate',
         'poverty_level']].sort_values('tpt_rank')

## Bagian 2.6.1 - Konsolidasi File Bersih

Cell berikut mengonfirmasi file hasil pembersihan dan transformasi yang tersedia di `data/clean/`.

In [ ]:
pd.DataFrame({
    'file': sorted(os.listdir(CLEAN_DIR))
})

---

# Grafik 1: Tren Kemiskinan Pulau Jawa 2021-2025 <a name="5"></a>

---

Grafik garis ini memperlihatkan perkembangan persentase penduduk miskin di enam provinsi Pulau Jawa dari 2021 hingga 2025. Jawa Barat disorot dengan warna merah dan ketebalan garis yang lebih besar, sedangkan provinsi lain ditampilkan dalam abu-abu sebagai konteks.

In [ ]:
df_plot = ppo_jawa.set_index('Provinsi').T
df_plot.index = df_plot.index.astype(int)

PROV_STYLE = {
    'JAWA BARAT'    : ('Jawa Barat',    '#E63946', 2.8, 7, 5),
    'DI YOGYAKARTA' : ('DI Yogyakarta', '#888888', 1.4, 4, 2),
    'JAWA TENGAH'   : ('Jawa Tengah',   '#888888', 1.4, 4, 2),
    'JAWA TIMUR'    : ('Jawa Timur',    '#888888', 1.4, 4, 2),
    'BANTEN'        : ('Banten',        '#888888', 1.4, 4, 2),
    'DKI JAKARTA'   : ('DKI Jakarta',   '#888888', 1.4, 4, 2),
}

fig, ax = plt.subplots(figsize=(11, 4.5))

for prov, (label, color, lw, ms, zo) in PROV_STYLE.items():
    is_jabar = prov == 'JAWA BARAT'
    ax.plot(df_plot.index, df_plot[prov],
            color=color, linewidth=lw, marker='o',
            markersize=ms, zorder=zo)
    ax.text(
        df_plot.index[-1] + 0.08, df_plot[prov].iloc[-1],
        label, color=color, fontsize=9, va='center',
        fontweight='bold' if is_jabar else 'normal',
    )

for ref_y in [4.0, 11.0]:
    ax.axhline(ref_y, color='#CCCCCC', linewidth=0.8, linestyle=':', zorder=0)
    ax.text(df_plot.index[0] - 0.12, ref_y,
            f'{ref_y:.1f}%', fontsize=13, color='#999999', va='center', ha='right')

for year in df_plot.index:
    ax.axvline(year, color='#DDDDDD', linewidth=0.8, linestyle=':', zorder=0)

ax.set_xlim(df_plot.index[0] - 0.1, df_plot.index[-1] + 0.75)
ax.set_ylim(2.8, 11.8)
ax.set_xticks(df_plot.index)
ax.set_xticklabels([str(y) for y in df_plot.index], fontsize=13, color='#555555')
ax.tick_params(axis='x', length=0)
ax.tick_params(axis='y', left=False, labelleft=False)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'g1_tren.png', dpi=150, bbox_inches='tight', facecolor='none')
plt.show()

#### Wawasan

> Secara keseluruhan, seluruh provinsi di Pulau Jawa mencatat penurunan angka kemiskinan dari 2021 ke 2025, meskipun beberapa provinsi mengalami fluktuasi di tengah periode. Jawa Tengah dan Jawa Timur bahkan mencatat sedikit kenaikan di tahun 2025, mengindikasikan bahwa pemulihan pasca 2021 tidak bersifat linier. Jawa Barat berada di posisi menengah dengan kisaran 6,65-7,52%, di bawah DI Yogyakarta dan Jawa Tengah yang konsisten di atas 8%, tetapi lebih tinggi dibandingkan DKI Jakarta dan Banten.

---

# Grafik 2: Peta Sebaran Kemiskinan Jawa Barat 2025 <a name="6"></a>

---

Peta koropleth ini menampilkan tingkat kemiskinan di setiap kabupaten/kota Jawa Barat pada 2025. Batas wilayah menggunakan data GADM 4.1 yang dibaca dengan `geopandas`. Wilayah tanpa padanan data (Pangandaran tidak tercakup dalam GADM 4.1 dan Waduk Cirata merupakan badan air) ditampilkan dalam abu-abu.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

gdf = gpd.read_file(GADM_PATH)
gdf_jabar = gdf[gdf['NAME_1'] == 'JawaBarat'].copy()

gdf_jabar = gdf_jabar.merge(
    jabar[['gadm_name', 'region', 'type', 'poverty_rate']],
    left_on='NAME_2', right_on='gadm_name',
    how='left',
)

ranked = (
    jabar[['gadm_name', 'region', 'poverty_rate']]
    .sort_values('poverty_rate', ascending=False)
    .reset_index(drop=True)
)
ranked['num'] = ranked.index + 1
gdf_jabar = gdf_jabar.merge(ranked[['gadm_name', 'num']], on='gadm_name', how='left')

reds_light = LinearSegmentedColormap.from_list(
    'Reds_light', plt.cm.Reds(np.linspace(0.05, 0.78, 256))
)

fig, ax_map = plt.subplots(figsize=(18, 8))
fig.subplots_adjust(left=0.16, right=0.92, top=0.96, bottom=0.04)

gdf_jabar.plot(
    column='poverty_rate',
    cmap=reds_light,
    linewidth=0.6,
    edgecolor='white',
    legend=True,
    legend_kwds={'label': 'Kemiskinan (%)', 'orientation': 'vertical', 'shrink': 0.6},
    missing_kwds={'color': '#DDDDDD', 'label': 'Tidak ada data'},
    ax=ax_map,
)

pos = {}
for _, row in gdf_jabar.iterrows():
    if pd.notna(row.get('num')):
        c = row.geometry.centroid
        pos[int(row['num'])] = (c.x, c.y, row.geometry.area)

NUDGE = {
    19: ( 0.025, -0.02),
    21: ( 0.15,   0.01),
    12: ( 0.1,   -0.1 ),
}

for num, (x, y, _) in pos.items():
    dx, dy = NUDGE.get(num, (0, 0))
    ax_map.annotate(
        str(num), xy=(x + dx, y + dy),
        ha='center', va='center',
        fontsize=9, fontweight='bold', color='#111111',
    )

ax_map.axis('off')

from matplotlib.patches import FancyBboxPatch
lh = 11 * 1.5 / (8 * 72)
x0, y0 = 0.03, 0.82
n_lines = len(ranked)
total_h = n_lines * lh
rect = FancyBboxPatch(
    (x0 - 0.005, y0 - total_h - 0.004),
    0.158, total_h + 0.014,
    transform=fig.transFigure,
    boxstyle='round,pad=0.002',
    facecolor='white', edgecolor='#CCCCCC',
    linewidth=1, alpha=0.92, zorder=2,
)
fig.add_artist(rect)
for i, (_, r) in enumerate(ranked.iterrows()):
    num = int(r['num'])
    line = f"{num:2d}  {r['region']:<18} {r['poverty_rate']:.2f}%"
    c  = '#8B1A1A' if num == 1 else ('#C0706F' if num == 27 else '#444444')
    fw = 'bold' if num in (1, 27) else 'normal'
    fig.text(x0, y0 - i * lh, line, fontsize=11, fontfamily='monospace',
             color=c, va='top', fontweight=fw, zorder=3)

plt.savefig(OUTPUT_DIR + 'g2_peta.png', dpi=150, bbox_inches='tight', pad_inches=0.2, facecolor='none')
plt.show()

#### Wawasan

> Kemiskinan di Jawa Barat terkonsentrasi di bagian selatan dan timur. Kabupaten Indramayu, Kuningan, Majalengka, dan Tasikmalaya memperlihatkan angka kemiskinan tertinggi yang melampaui 10%. Sebaliknya, kota-kota di koridor utara seperti Kota Depok, Kota Bekasi, dan Kota Bandung mencatat tingkat kemiskinan terendah di bawah 5%, mencerminkan konsentrasi aktivitas ekonomi formal di kawasan tersebut.

---

# Grafik 3: Korelasi Kemiskinan dan Pengangguran 2025 <a name="7"></a>

---

Grafik pencar ini menguji hubungan antara persentase penduduk miskin dan tingkat pengangguran terbuka (TPT) di 27 kabupaten/kota Jawa Barat pada 2025. Kota dan Kabupaten dibedakan dengan warna berbeda, dan garis regresi linier ditampilkan untuk masing-masing kelompok.

In [ ]:
COLOR_MAP = {'Kota': '#1A6B8A', 'Kabupaten': '#C0392B'}

fig, ax = plt.subplots(figsize=(5.5, 3.5))

for t, grp in jabar.groupby('type'):
    ax.scatter(
        grp['unemployment_rate'], grp['poverty_rate'],
        color=COLOR_MAP[t], s=65, alpha=0.85,
        edgecolors='white', linewidths=0.5,
        label=t, zorder=3,
    )
    slope, intercept, *_ = stats.linregress(
        grp['unemployment_rate'], grp['poverty_rate']
    )
    xl = np.linspace(jabar['unemployment_rate'].min(),
                     jabar['unemployment_rate'].max(), 100)
    ax.plot(xl, slope * xl + intercept,
            color=COLOR_MAP[t], linewidth=1.8, alpha=0.65)

LABEL_OFFSET = {
    'Bekasi':     (6, 6),
    'Kota Cimahi': (6, -10),
}
for _, row in jabar[jabar['region'].isin(ANOMALI_ALL)].iterrows():
    dx, dy = LABEL_OFFSET.get(row['region'], (6, 3))
    ax.annotate(
        row['region'],
        xy=(row['unemployment_rate'], row['poverty_rate']),
        xytext=(dx, dy), textcoords='offset points',
        fontsize=8, color='#333333',
    )

r, _ = stats.pearsonr(jabar['unemployment_rate'], jabar['poverty_rate'])
ax.text(0.97, 0.96, f'r = {r:.2f}', transform=ax.transAxes,
        fontsize=10, va='top', ha='right', color='#555555')

ax.legend(title='Tipe Wilayah', frameon=True, fontsize=10,
          loc='lower left', borderaxespad=0.5)
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))
ax.tick_params(axis='x', labelsize=8, colors='#777777')
ax.tick_params(axis='y', labelsize=8, colors='#777777')
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'g3_scatter.png', dpi=150, bbox_inches='tight', facecolor='none')
plt.show()

#### Wawasan

> Korelasi antara kemiskinan dan pengangguran di Jawa Barat lemah dan negatif (r = -0.37), menunjukkan bahwa tingginya pengangguran tidak selalu sejalan dengan tingginya kemiskinan. Kabupaten Pangandaran, Tasikmalaya, dan Ciamis memiliki tingkat pengangguran yang sangat rendah tetapi angka kemiskinan tinggi di Jawa Barat, mengindikasikan bahwa pekerjaan yang tersedia di kawasan tersebut belum mampu mengangkat daya beli masyarakat.

---

# Grafik 4: Kota-Kota Anomali Jawa Barat 2025 <a name="8"></a>

---

Untuk menguji apakah garis kemiskinan menjelaskan anomali, setiap wilayah anomali dipetakan ke dua persentil secara bersamaan: persentil TPT dan persentil garis kemiskinan relatif terhadap 27 wilayah Jawa Barat. Apabila garis kemiskinan memang menjadi penyebab anomali, wilayah dengan TPT rendah seharusnya memiliki garis kemiskinan yang tinggi pula, dan sebaliknya. Kemiringan garis setiap wilayah langsung memperlihatkan apakah hipotesis itu terkonfirmasi atau tidak.

In [ ]:
jabar['tpt_rank'] = jabar['unemployment_rate'].rank(method='min').astype(int)
jabar['gk_rank']  = jabar['poverty_level'].rank(method='min').astype(int)

anomali = jabar[jabar['region'].isin(ANOMALI_ALL)].copy()

C_HIGH = '#E63946'
C_LOW  = '#2A9D8F'

fig, ax = plt.subplots(figsize=(5.5, 4.8))

for grp, color in [(ANOMALI_HIGH, C_HIGH), (ANOMALI_LOW, C_LOW)]:
    subset = anomali[anomali['region'].isin(grp)].sort_values('tpt_rank')
    for i, (_, row) in enumerate(subset.iterrows()):
        tpt, gk = row['tpt_rank'], row['gk_rank']
        ax.plot([0, 1], [tpt, gk], color=color, lw=2.2,
                ls='-', alpha=0.88)
        ax.scatter([0, 1], [tpt, gk], color=color, s=60, zorder=5)
        ax.text(-0.04, tpt, row['region'],
                ha='right', va='center', fontsize=8, color=color)

ax.axhline(14, color='#AAAAAA', lw=0.9, ls='--')
ax.text(0.5, 14.5, 'median', ha='center', va='bottom',
        fontsize=7, color='#AAAAAA')

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0],[0], color=C_LOW,  lw=2,
           label='Pengangguran Tinggi, Kemiskinan Rendah'),
    Line2D([0],[0], color=C_HIGH, lw=2,
           label='Pengangguran Rendah, Kemiskinan Tinggi'),
]
ax.legend(handles=legend_el, loc='lower center',
          bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8, frameon=True)
plt.tight_layout(rect=[0, 0.16, 1, 1])

ax.set_xticks([])
ax.set_xlim(-0.18, 1.08)
ax.set_ylim(0, 30)
for ref_y in [1, 27]:
    ax.axhline(ref_y, color='#CCCCCC', linewidth=0.8, linestyle=':', zorder=0)
for x_val in [0, 1]:
    ax.axvline(x_val, color='#DDDDDD', linewidth=0.8, linestyle=':', zorder=0)
ax.tick_params(axis='y', left=False, labelleft=False)
ax.set_yticks([])
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.2, color='#DDDDDD')
plt.savefig(OUTPUT_DIR + 'g4_slopegraph.png', dpi=150, bbox_inches='tight', facecolor='none')
plt.show()

#### Wawasan

> Tiga wilayah rasio tinggi (merah) memiliki TPT di persentil terbawah tetapi garis kemiskinan yang hanya berada di kisaran menengah ke bawah. Garis mereka condong ke atas atau relatif datar, artinya garis kemiskinan tidak tinggi sehingga hipotesis biaya hidup tidak terkonfirmasi. Kemiskinan tinggi di sini bukan karena standar subsisten yang mahal, melainkan karena upah sektor pertanian memang rendah secara absolut.

> Tiga wilayah rasio rendah (hijau) memiliki garis kemiskinan di persentil tertinggi, jauh di atas persentil TPT mereka. Garis mereka menanjak tajam ke kanan, mengonfirmasi bahwa tingginya standar biaya hidup memang berkorelasi dengan rendahnya kemiskinan meski penganggurannya tinggi.